# Task 1: News Topic Classifier Using BERT
### DevelopersHub Corporation — AI/ML Engineering Advanced Internship

**Author:** Sadaf
**Internship:** AI/ML Engineering Internship – DevelopersHub Corporation

---

## Problem Statement & Objective

News platforms publish thousands of articles every day, and manually sorting them into topics (World, Sports, Business, Sci/Tech) is slow and inefficient.

**Objective:** Fine-tune a pre-trained Transformer model (`bert-base-uncased`) to automatically classify news headlines into one of **4 topic categories** using the **AG News Dataset**, evaluate it using **Accuracy** and **F1-score**, and deploy it as a **live interactive demo** using Gradio.

### Workflow Overview
1. Load & explore the AG News dataset
2. Tokenize and preprocess text using BERT tokenizer
3. Fine-tune `bert-base-uncased` using Hugging Face `Trainer` API
4. Evaluate using Accuracy & F1-score + Confusion Matrix
5. Save the fine-tuned model
6. Deploy a live demo with Gradio
7. Final summary & insights

## 1 Install & Import Libraries

In [ ]:
# Install required libraries (Colab usually has most pre-installed, this just ensures correct versions)
!pip install -q transformers datasets evaluate scikit-learn gradio accelerate -U

In [ ]:
# Import all required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Check if GPU is available (Colab: Runtime > Change runtime type > GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Using device: {device}")

## 2 Dataset Loading & Preprocessing

We use the **AG News Dataset**, available directly from the Hugging Face `datasets` library. It contains news headlines + short descriptions labeled into **4 classes**:

| Label | Category |
|-------|----------|
| 0 | World |
| 1 | Sports |
| 2 | Business |
| 3 | Sci/Tech |

In [ ]:
# Load the AG News dataset directly from Hugging Face Hub
# NOTE: the old short repo name "ag_news" has been moved/deprecated on the Hub.
# Newer huggingface_hub versions require the full 'namespace/name' path — the current working id is "fancyzhx/ag_news".
dataset = load_dataset("fancyzhx/ag_news")
print(dataset)

In [ ]:
# Peek at a few samples
for i in range(3):
    print(f"Text: {dataset['train'][i]['text']}")
    print(f"Label: {dataset['train'][i]['label']}")
    print("-" * 60)

In [ ]:
# For faster training on Colab's free GPU, we take a stratified-ish subset.
# For FULL MARKS / production use, simply increase these numbers (or remove .select() entirely)
# to train on the full 120,000 train / 7,600 test samples.
TRAIN_SIZE = 8000
TEST_SIZE = 2000

train_dataset = dataset["train"].shuffle(seed=42).select(range(TRAIN_SIZE))
test_dataset = dataset["test"].shuffle(seed=42).select(range(TEST_SIZE))

print(f" Train samples: {len(train_dataset)}")
print(f" Test samples: {len(test_dataset)}")

In [ ]:
# Visualize class distribution to confirm balanced classes
label_names = ["World 🌍", "Sports ⚽", "Business 💼", "Sci/Tech 🔬"]

train_labels = train_dataset["label"]
plt.figure(figsize=(7,4))
sns.countplot(x=train_labels, palette="viridis")
plt.xticks(ticks=[0,1,2,3], labels=label_names)
plt.title(" Class Distribution in Training Set")
plt.xlabel("News Category")
plt.ylabel("Count")
plt.show()

### Tokenization
We use `bert-base-uncased`'s tokenizer to convert raw text into token IDs, attention masks, and padded/truncated sequences BERT can understand.

In [ ]:
# Load BERT tokenizer
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

# Tokenization function: pads/truncates all sequences to a fixed max length
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=64 # headlines are short, so 64 tokens is sufficient
    )

# Apply tokenization to train & test sets
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

print(" Tokenization complete!")
print(train_dataset[0])

## 3 Model Development & Training

We fine-tune `bert-base-uncased` with a classification head for **4 output classes** using the Hugging Face `Trainer` API.

In [ ]:
# Load pre-trained BERT with a classification head (4 labels)
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
).to(device)

print(" Model loaded successfully!")

In [ ]:
# Define evaluation metrics: Accuracy & F1-score (weighted, since we may have class imbalance)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {"accuracy": acc, "f1": f1}

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./bert_news_classifier",
    eval_strategy="epoch", # evaluate at the end of every epoch
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3, # 3 epochs is a good starting point for fine-tuning BERT
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none" # disable wandb/other loggers in Colab
)

In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print(" Trainer ready! Starting fine-tuning... ")

In [ ]:
# Fine-tune BERT on AG News (this may take a few minutes on Colab GPU)
train_result = trainer.train()

In [ ]:
# Plot training loss curve from Trainer's log history
logs = trainer.state.log_history
train_loss = [log["loss"] for log in logs if "loss" in log]
steps = list(range(len(train_loss)))

plt.figure(figsize=(8,4))
plt.plot(steps, train_loss, color="crimson", marker="o", markersize=3)
plt.title(" Training Loss Over Steps")
plt.xlabel("Logging Step")
plt.ylabel("Loss")
plt.grid(alpha=0.3)
plt.show()

## 4 Evaluation

We evaluate the fine-tuned model on the held-out test set using **Accuracy**, **F1-score**, a **classification report**, and a **confusion matrix**.

In [ ]:
# Evaluate on test set
eval_results = trainer.evaluate()
print(" Evaluation Results:")
for k, v in eval_results.items():
    print(f" {k}: {v:.4f}")

In [ ]:
# Get predictions for detailed analysis
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

# Classification report (precision, recall, F1 per class)
print(" Classification Report:\n")
print(classification_report(true_labels, preds, target_names=label_names))

In [ ]:
# Confusion Matrix Heatmap
cm = confusion_matrix(true_labels, preds)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_names, yticklabels=label_names)
plt.title(" Confusion Matrix — BERT News Classifier")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

In [ ]:
# Final summary metrics table
final_acc = accuracy_score(true_labels, preds)
final_f1 = f1_score(true_labels, preds, average="weighted")

results_df = pd.DataFrame({
    "Metric": ["Accuracy ", "Weighted F1-score "],
    "Score": [f"{final_acc:.4f}", f"{final_f1:.4f}"]
})
results_df

## 5 Save the Fine-Tuned Model

In [ ]:
# Save model & tokenizer locally (so it can be reloaded for deployment)
SAVE_PATH = "./bert_news_classifier_final"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f" Model & tokenizer saved to {SAVE_PATH}")

## 6 Deployment — Live Interactive Demo with Gradio

We deploy the fine-tuned model with **Gradio**, so anyone can type a news headline and instantly see the predicted category live inside this notebook!

In [ ]:
import gradio as gr

# Reload model & tokenizer from saved path (simulates a real deployment scenario)
inference_model = BertForSequenceClassification.from_pretrained(SAVE_PATH).to(device)
inference_tokenizer = BertTokenizerFast.from_pretrained(SAVE_PATH)
inference_model.eval()

# Prediction function
def classify_news(headline):
    inputs = inference_tokenizer(
        headline,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=64
    ).to(device)

    with torch.no_grad():
        outputs = inference_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]

    return {label_names[i]: float(probs[i]) for i in range(4)}

# Build Gradio interface
demo = gr.Interface(
    fn=classify_news,
    inputs=gr.Textbox(lines=2, placeholder="📰 Type a news headline here... e.g. 'Apple unveils new AI chip for iPhones'"),
    outputs=gr.Label(num_top_classes=4),
    title="📰🤖 BERT News Topic Classifier",
    description="Fine-tuned bert-base-uncased on AG News — classifies headlines into World 🌍, Sports ⚽, Business 💼, or Sci/Tech 🔬",
    examples=[
        "Stocks tumble as investors worry about inflation 💼",
        "Local team wins championship in dramatic overtime finish ⚽",
        "NASA announces new mission to explore Europa's icy surface 🔬",
        "World leaders meet at UN summit to discuss climate policy 🌍"
    ]
)

# Launch (set share=True to get a public link from Colab)
demo.launch(share=True, debug=False)

## 7 Final Summary & Insights

### What we did
- Loaded and explored the **AG News dataset** (4 balanced classes: World , Sports , Business , Sci/Tech )
- Tokenized headlines using the `bert-base-uncased` tokenizer
- Fine-tuned `bert-base-uncased` for 3 epochs using the Hugging Face `Trainer` API
- Evaluated performance using **Accuracy**, **weighted F1-score**, a classification report, and a confusion matrix
- Saved the fine-tuned model & tokenizer for reuse
- Deployed a **live interactive Gradio demo** for real-time predictions

### Key Observations
- Fine-tuned BERT achieves strong accuracy and F1-score on news topic classification, showing the power of **transfer learning** — a general-purpose language model adapts very well to a specific downstream task with just a few epochs.
- The confusion matrix shows most misclassifications happen between semantically overlapping categories (e.g., **Business vs Sci/Tech **, since tech-business news like "company earnings" can straddle both).
- Training on a subset (8,000 samples) already yields solid results; using the **full 120,000-sample training set** (simply remove the `.select()` limit) would likely push accuracy/F1 even higher.

### Possible Improvements
- Train on the **full dataset** for more epochs with a learning-rate scheduler
- Try `distilbert-base-uncased` for a faster, lighter deployment-friendly model
- Add **early stopping** based on validation F1 to avoid overfitting
- Experiment with **max_length** tuning if longer article bodies (not just headlines) are used

---
### Skills Demonstrated
 NLP using Transformers &nbsp;|&nbsp; Transfer Learning & Fine-Tuning &nbsp;|&nbsp; Evaluation Metrics for Text Classification &nbsp;|&nbsp; Lightweight Model Deployment (Gradio)

---
*Task 1 completed as part of the AI/ML Engineering Advanced Internship at DevelopersHub Corporation *